In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (2), incompatible header (7))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (4), incompatible header (14))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


# Axisymmetric Linear Elasticity

This example demonstrates the solution of a two-dimensional axisymmetric
linear elasticity problem using LowLevelFEM.

The problem is solved in two different ways:

1. using the built-in axisymmetric elasticity solver,
2. using the weak-form DSL.

The two solutions are then compared to verify the correctness of the
DSL implementation.

In [2]:
structured_rect_mesh(x0=20, lx=80, ly=20, n=80, order=2)
mat = Material("body");

## Geometry and material

A quadratic structured mesh is generated for a hollow cylinder.
The inner radius avoids the singularity associated with the axis of
revolution.

Linear isotropic elastic material properties are assigned to the body.

## Built-in axisymmetric formulation

First, the problem is solved using the dedicated axisymmetric elasticity
implementation available in LowLevelFEM.

The right boundary is subjected to a radial traction, while the lower-left
corner is constrained to remove rigid body motion.

In [3]:
prob = Problem([mat], type=:AxiSymmetric)

ld = load("left", fx=1);

In [4]:
bc = BoundaryCondition("leftbottom", uy=0);

In [5]:
u1 = solveDisplacement(prob, load=[ld], support=[bc]);

In [51]:
@time K1 = stiffnessMatrix(prob)
K1[:, :]

  1.443454 seconds (11.37 M allocations: 2.755 GiB, 28.64% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6565404 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [7]:
S1 = solveStress(u1);

In [8]:
showDoFResults(u1, name="u1", visible=true, factor=1e5)
showStressResults(S1, name="σ1 - von Mises");

## Reference solution

The displacement and von Mises stress obtained from the built-in
axisymmetric formulation will be used as the reference solution.

## Weak-form DSL formulation

The same problem is assembled using the LowLevelFEM weak-form DSL.

The axisymmetric strain-displacement operator is written directly from
its mathematical definition.

In [9]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u);

In [10]:
r = ScalarField(Pu, "body", (x, y, z)->x);

### Axisymmetric strain operator

The strain vector is written as

$$
\varepsilon =
\begin{bmatrix}
\partial u_r/\partial r \\
u_r/r \\
\partial u_z/\partial z \\
\partial u_r/\partial z + \partial u_z/\partial r
\end{bmatrix},
$$

which is represented as

$$
\varepsilon = A_1 \nabla^s u + A_2 u.
$$

This decomposition allows the operator to be expressed naturally in the DSL.

In [11]:
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]

4×3 Matrix{Int64}:
 1  0  0
 0  0  0
 0  1  0
 0  0  1

In [12]:
A2 = [0 0; 1/r 0; 0 0; 0 0];

In [13]:
B = full(A1 ⋅ SymGrad(Pu)) + reduced(A2 ⋅ Pu);

In [14]:
E = mat.E
ν = mat.ν

D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

4×4 Matrix{Float64}:
 2.69231e5  1.15385e5  1.15385e5      0.0
 1.15385e5  2.69231e5  1.15385e5      0.0
 1.15385e5  1.15385e5  2.69231e5      0.0
 0.0        0.0        0.0        76923.1

### System assembly

The global stiffness matrix is assembled from the weak form

$$
K = \int_\Omega B^\mathrm{T} D B \; 2\pi r \; d\Omega.
$$

The factor $2\pi r$ accounts for the axisymmetric volume measure.

In [41]:
GC.gc()

In [55]:
@time K2 = ∫(B' ⋅ D ⋅ B * (2π*r))
K2[:, :]

  4.804996 seconds (33.46 M allocations: 2.239 GiB, 7.50% gc time)


206402×206402 SparseArrays.SparseMatrixCSC{Float64, Int64} with 6566377 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⠿⣿⣛⡛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠛⠲⠦⣤⣀⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠒⠦⢤⣀⡀⠀⎥
⎢⣿⡆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⎥
⎢⣿⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢻⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠸⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢻⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [16]:
f2 = ∫(Pu ⋅ [1, 0] * (2π*r), Γ="left");

In [17]:
using Profile

# Compile and initialize all relevant methods.
K2_warmup = ∫(B' ⋅ D ⋅ B * (2π * r))

GC.gc()
Profile.clear()

@time begin
    @profile K2_profiled = ∫(B' ⋅ D ⋅ B * (2π * r))
end

Profile.print(
    format=:flat,
    sortedby=:count,
    mincount=10,
    C=false
)

  6.410038 seconds (33.51 M allocations: 2.241 GiB, 9.37% gc time, 7.03% compilation time)
 Count  Overhead File                    Line Function
 =====  ======== ====                    ==== ========
    10         3 @LowLevelFEM]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\/src/multifield.jl]8;;\ 1248 #detect_pattern#487
    11         0 @LinearAlgebra]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.7%2B0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/matmul.jl\/src/matmul.jl]8;;\   18 dot
    11         0 @LinearAlgebra]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.7%2B0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/blas.jl\/src/blas.jl]8;;\  396 dot
    11        11 @LinearAlgebra]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.7%2B0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/blas.jl\/src/blas.jl]8;;\  346 dot

In [18]:
Profile.print(
    format=:tree,
    sortedby=:count,
    mincount=10,
    C=false
)

Overhead ╎ [+additional indent] Count File:Line  Function
  33╎33    ]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\@LowLevelFEM]8;;\]8;;file://perebal-Latitude-5580/home/perebal/Dokumentumok/GitHub/LowLevelFEM.jl/src/multifield.jl\/…field.jl:1269]8;;\  #detect_pattern!#488
  42╎42    ]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.7%2B0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/blas.jl\@LinearAlgebra]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/juliaup/julia-1.12.7%2B0.x64.linux.gnu/share/julia/stdlib/v1.12/LinearAlgebra/src/blas.jl\/…las.jl:644]8;;\  gemv!
    ╎356   ]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/stdio.jl\@IJulia]8;;\]8;;file://perebal-Latitude-5580/home/perebal/.julia/packages/IJulia/Vl5w1/src/stdio.jl\/src/stdio.jl:274]8;;\  (::IJulia.var"#watch_stdio##6#watch_stdio#…
    ╎ 356   ]8;;file://perebal-Latitude-

## Solution

The linear system assembled from the DSL is solved using the same boundary
conditions as the reference formulation.

In [19]:
@time u2 = solveField(K2, f2, support=[bc]);

  5.409272 seconds (796 allocations: 1.137 GiB, 6.12% gc time)


In [20]:
@time u2 = solveField(Symmetric(K2), f2, support=[bc]);

  5.260085 seconds (5.29 M allocations: 1.074 GiB, 5.31% gc time, 64.94% compilation time)


In [21]:
@time u3 = solveField(K2, f2, support=[bc], iterative=true, reltol=1e-4, maxiter=100);

  2.035524 seconds (834.20 k allocations: 230.055 MiB, 0.37% gc time, 23.47% compilation time)


In [22]:
@time u3 = solveField(Symmetric(K2), f2, support=[bc], iterative=true, reltol=1e-4, maxiter=100);

  3.263794 seconds (408.76 k allocations: 211.755 MiB, 6.93% gc time, 24.45% compilation time)


In [23]:
showDoFResults(u2, name="u2", visible=true, factor=1e5);

## Stress recovery

The strain components are computed from the displacement field,
followed by Hooke's law to obtain the stress tensor.

Finally, the von Mises equivalent stress is evaluated.

In [24]:
εr = ∂x(u2[1])
εφ = u2[1] / r
εz = ∂y(u2[2])
γrz = ∂y(u2[1]) + ∂x(u2[2]);

In [25]:
ε = [εr, εφ, εz, γrz];

In [26]:
σ = D * ε;

In [27]:
σr = σ[1]
σφ = σ[2]
σz = σ[3]
τrz = σ[4];

In [28]:
σeqv2 = √(((σr - σφ)^2 + (σφ - σz)^2 + (σz - σr)^2) / 2 + 3τrz^2);

In [29]:
showElementResults(σeqv2, name="σ2 - von Mises");

## Verification

The displacement and stress fields obtained from the weak-form DSL agree
with those produced by the built-in axisymmetric formulation, confirming
the correctness of the DSL implementation.

In [30]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
